# Lab 10: Correlation & Causality — Diagnosing Bad Controls
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnostic Lab | 78 min Core + 35 min AI + 45 min Extension

---

**Learning Objectives:**
- Diagnose an estimate that got worse when a control was added, and name the structure responsible
- Show that the pre-treatment heuristic fails, and state what replaces it
- Separate a total effect from a direct effect, and say which one the question asked for
- Read and run a graph-driven adjustment-set finder (`src/dag_utils.py`) instead of choosing controls by hand
- Check an adjustment set proposed by AI against a DAG, and revise the prompt that produced it

**Dataset:** Simulated DGPs with known truth, plus LaLonde NSW + CPS controls

**Format:** Each part hands you a regression that is wrong and asks what happened before it
tells you. Write your diagnosis before running the reveal cell.

**Deliverable:** your four written diagnoses, your four answers on the AI's adjustment set,
and the `src/dag_utils.py` file the notebook writes.

**Time:** Core (78 min) + AI (35 min) + Extension (optional, 45 min) + Challenge (optional, ungraded)

With setup, the README and the GitHub submission, plan on about two and a quarter hours, most of it after class. The optional Extension adds about 45 minutes.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

---

> ### What you write, and what you run
>
> **You write, in text cells:**
> - **Diagnoses 1, 2 and 3, and the LaLonde case:** your diagnosis of each, written before you run the reveal where there is one: four, three, three and two short answers. A third LaLonde question is optional and not graded.
> - **The AI Expansion:** you revise the given prompt once, type each set the AI proposes into a cell as a one-line Python list, and write four short answers about what the AI got wrong, judged against the DAG.
> - **Extension (optional):** three interpretation answers.
>
> **You run and read:** every code cell in the diagnoses, the LaLonde case and Ship It, and the AI section's setting and check cells. They are the analyst's regressions, the reveals, and the finished `dag_utils.py` module, which is working code to learn from, not to retype. Read each cell before you run it: the questions ask about what it prints.
>
> **Code you write yourself (none of it required this week):**
> - the Extension's sensitivity analysis: one function body of about six lines, then a sweep and a chart;
> - the take-home Challenge, a DAG of your own run through `dag_utils`, from a five-step outline.

In [ ]:
# GIVEN — run as-is
# Setup: libraries, folders and the random generator
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import statsmodels.api as sm

Path("src").mkdir(exist_ok=True)       # Ship It writes its module into this folder
Path("figures").mkdir(exist_ok=True)   # for any chart you save in the Extension

RANDOM_STATE = 42
rng = np.random.default_rng(seed=RANDOM_STATE)   # Diagnoses 1-3 draw from this one generator, in order
print("Setup complete.")

## Diagnosis 1: The Control That Made It Worse (25 min)

An analyst regresses Y on D and gets a null. Uncomfortable with a null, they add the richest
covariate available and get a large, significant coefficient. They report the second one, on
the grounds that it controls for more.

The true effect is zero.

Run Diagnoses 1-3 once, in order: they share one random generator, so re-running a cell
changes the numbers.

**Reading the code.** `sm.add_constant(d)` adds a column of 1s, so the regression has an
intercept; `np.column_stack([d, c])` puts `d` and `c` side by side as two columns. After the
fit, `.params[1]`, `.bse[1]` and `.pvalues[1]` are the coefficient, standard error and
p-value of the first variable after the constant, which is D. In the prints, `{x:<24}`
left-aligns in 24 characters, `{x:>10.3f}` right-aligns in 10 with three decimals, and
`"-" * 61` is a line of 61 dashes. `10_000` is 10000: the underscore only helps you read it.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# The analyst's two specifications
n = 10_000
d = rng.normal(0, 1, n)
y = rng.normal(0, 1, n)                       # drawn independently of d
c = 1.2 * d + 1.2 * y + rng.normal(0, 1, n)   # rich, well-measured, available

spec1 = sm.OLS(y, sm.add_constant(d)).fit()
spec2 = sm.OLS(y, sm.add_constant(np.column_stack([d, c]))).fit()

print(f"{'specification':<24}{'estimate':>10}{'std err':>10}{'p':>9}{'R^2':>8}")
print("-" * 61)
print(f"{'Y ~ D':<24}{spec1.params[1]:>10.3f}{spec1.bse[1]:>10.3f}"
      f"{spec1.pvalues[1]:>9.3f}{spec1.rsquared:>8.3f}")
print(f"{'Y ~ D + C':<24}{spec2.params[1]:>10.3f}{spec2.bse[1]:>10.3f}"
      f"{spec2.pvalues[1]:>9.3f}{spec2.rsquared:>8.3f}")

### YOUR DIAGNOSIS

1. The second specification has far higher R-squared and a much smaller p-value. Explain why both of those facts are irrelevant to whether it is the better estimate.
2. Name the structure. Draw the three-node graph and mark which node is being conditioned on.
3. Give the sign of the manufactured association and derive it from the DGP — do not just read it off the output.
4. Construct a variant where the manufactured association goes the *other* way, and say what you changed.

*Your diagnosis here:*

In [ ]:
# GIVEN — run as-is
# The reveal: hold C nearly fixed and compare D and Y again
# band is True for the units whose C lies within 0.35 of its median
band = np.abs(c - np.median(c)) < 0.35
# d[band] keeps the entries of d where band is True, and y[band] the same units of y

# np.corrcoef returns a 2 x 2 matrix; [0, 1] picks the correlation off the diagonal
print(f"Correlation of D and Y, everyone:        {np.corrcoef(d, y)[0,1]: .3f}")
print(f"Correlation of D and Y, holding C fixed: {np.corrcoef(d[band], y[band])[0,1]: .3f}")

**Reading the reveal.** Within a slice of C, a high D must be met by a low Y to keep C where
it is. Regression control is this slicing, done smoothly and for every value of C at once.

---

## Diagnosis 2: The Rule That Does Not Hold (16 min)

The analyst accepts the collider lesson and adopts the standard defence: *never control for
anything measured after treatment; anything pre-treatment is safe.*

Below, W is measured before treatment. It is not caused by D. It is not caused by Y.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# The analyst's two specifications, with W measured before treatment
n = 20_000
u1 = rng.normal(0, 1, n)          # unmeasured, drives treatment
u2 = rng.normal(0, 1, n)          # unmeasured, drives outcome
w = 1.0 * u1 + 1.0 * u2 + rng.normal(0, 1, n)      # measured FIRST

d = 1.0 * u1 + rng.normal(0, 1, n)
y = 1.0 * u2 + rng.normal(0, 1, n)                 # TRUE effect of D = 0

without_w = sm.OLS(y, sm.add_constant(d)).fit()
with_w = sm.OLS(y, sm.add_constant(np.column_stack([d, w]))).fit()

print(f"{'specification':<32}{'estimate':>10}{'p':>10}{'truth':>8}")
print("-" * 60)
print(f"{'Y ~ D':<32}{without_w.params[1]:>10.3f}{without_w.pvalues[1]:>10.3f}{0.0:>8.1f}")
print(f"{'Y ~ D + W   (pre-treatment)':<32}{with_w.params[1]:>10.3f}"
      f"{with_w.pvalues[1]:>10.3f}{0.0:>8.1f}")

### YOUR DIAGNOSIS

1. Write the path from D to Y that runs through W. Say which node on it is a collider, and why the path is blocked until W is conditioned on.
2. W is pre-treatment, so the timing rule permits it. State precisely what the timing rule *is* good for, and why no rule based on timing alone can be sufficient.
3. Neither u1 nor u2 is observed. Could you detect this problem from the data alone? Justify your answer — this determines whether M-bias is a testable concern or a purely structural one.

*Your diagnosis here:*

---

## Diagnosis 3: The Right Answer to the Wrong Question (11 min)

No bias this time. The regression below is correctly specified for *a* causal quantity. It is
simply not the quantity the analyst was asked for, and nothing in the output says so.

**Reading the code.** `rng.binomial(1, 0.5, n)` draws `n` zeros and ones, each a 1 with
probability 0.5: here, who gets training.

In [ ]:
# THE ANALYST'S CODE — run as-is, then diagnose it
# A training programme raises earnings, partly by raising skill
n = 20_000
d = rng.binomial(1, 0.5, n)                    # training
m = 1.5 * d + rng.normal(0, 1, n)              # skill gained
y = 1.0 * d + 2.0 * m + rng.normal(0, 1, n)    # earnings

TOTAL = 1.0 + 2.0 * 1.5      # direct 1.0 plus 2.0 x 1.5 through skill
DIRECT = 1.0

total_fit = sm.OLS(y, sm.add_constant(d)).fit()
ctrl_fit = sm.OLS(y, sm.add_constant(np.column_stack([d, m]))).fit()

print(f"{'specification':<26}{'estimate':>10}")
print("-" * 36)
print(f"{'Y ~ D':<26}{total_fit.params[1]:>10.3f}")
print(f"{'Y ~ D + M':<26}{ctrl_fit.params[1]:>10.3f}")
print(f"\nTrue effects: total {TOTAL:.1f}, direct {DIRECT:.1f}")

### YOUR DIAGNOSIS

1. Both regressions are unbiased for something. Say what each targets, and why "adding a control changed my estimate by a factor of four" is not evidence that either is wrong.
2. A policymaker asks: "should we fund this training programme?" Which number answers that question? Defend the choice.
3. Now suppose skill is measured with error. Argue what happens to the direct-effect estimate, and whether the error makes it closer to or further from the total effect.

*Your diagnosis here:*

---

## The Real Case: LaLonde (16 min)

LaLonde (1986) is the field's standing
test of whether observational adjustment can recover an experimental answer, because here
the experiment exists and can be checked against.

**Reading the code.** `pd.read_stata` reads a Stata `.dta` file.
`exp_df.loc[exp_df["treat"] == 1, "re78"]` keeps the rows where `treat` is 1, and only the
column `re78` (1978 earnings). `pd.concat([a, b], ignore_index=True)` stacks two tables and
renumbers the rows. `.astype(float)` turns every column into plain numbers, which is what
statsmodels expects. `{x:>9,.0f}` right-aligns in 9 characters, with thousands separators
and no decimals.

In [ ]:
# GIVEN — run as-is
# The experimental benchmark
# Dehejia & Wahba (1999) files, served by NBER.
#   nsw_dw.dta       the RANDOMISED experiment: 185 treated, 260 control
#   cps_controls.dta a non-experimental comparison group drawn from the CPS
EXPERIMENT = "https://users.nber.org/~rdehejia/data/nsw_dw.dta"
CPS = "https://users.nber.org/~rdehejia/data/cps_controls.dta"

exp_df = pd.read_stata(EXPERIMENT)
cps_df = pd.read_stata(CPS)

treated_re78 = exp_df.loc[exp_df["treat"] == 1, "re78"]
control_re78 = exp_df.loc[exp_df["treat"] == 0, "re78"]
benchmark = treated_re78.mean() - control_re78.mean()

print(f"Experimental benchmark (randomised): ${benchmark:>9,.0f}")

In [ ]:
# GIVEN — run as-is
# The observational alternative: the same treated men, compared with CPS men
obs = pd.concat([exp_df[exp_df["treat"] == 1], cps_df], ignore_index=True)
covs = ["age", "education", "black", "hispanic", "married", "nodegree",
        "re74", "re75"]

naive = (obs.loc[obs["treat"] == 1, "re78"].mean()
         - obs.loc[obs["treat"] == 0, "re78"].mean())
# ["treat"] + covs joins two lists: treat first, then the eight covariates
adj = sm.OLS(obs["re78"].astype(float),
             sm.add_constant(obs[["treat"] + covs].astype(float))).fit()

print(f"Observational, no adjustment:        ${naive:>9,.0f}")
# .params['treat'] picks the coefficient by its column name
print(f"Observational, full covariate set:   ${adj.params['treat']:>9,.0f}")
print()
print(f"Comparison group sizes: {len(control_re78)} experimental vs {len(cps_df):,} CPS")

**Verification checkpoint:** the experimental benchmark lands near \$1,800 (measured:
\$1,794), the naive observational estimate is large and *negative* (−\$8,498), and covariate
adjustment pulls it back most of the way (\$699).

### YOUR DIAGNOSIS

1. Adjustment moves the estimate a long way toward the benchmark. Explain why that is *not* sufficient evidence that the adjustment strategy is valid, referring to what you would and would not know without the experiment.
2. `re74` and `re75` are pre-treatment earnings. Draw the DAG for who enrols and what they earn, and argue the status of `re74`/`re75` in it. Are they confounders, proxies for a confounder, or something else — and does the answer change if you had only `re75`?
3. *Optional, not graded:* Dehejia and Wahba recovered the benchmark closely with propensity-score methods; Smith and Todd showed the result is fragile to the comparison group and specification. State what that disagreement implies for how you should report an observational estimate of your own.

*Your diagnosis here:*

---

## Ship It: `src/dag_utils.py` (10 min)

You have been classifying controls by eye. Once the graph is written down that classification
is mechanical, and a function can do it.

Here is the finished module. Read two functions closely: `path_blocked`, against the
blocking rules from the lecture (a non-collider in the set blocks a path; a collider blocks
it unless the set holds the collider or one of its descendants), and
`is_valid_adjustment_set`, which applies them to every backdoor path. Skim the rest. Then
run the two cells: the first writes the file, the
second imports it and tests it on two small graphs.

**Reading the module.** The first line, `%%writefile src/dag_utils.py`, saves the cell as
that file instead of running it. `G.has_edge(a, b)` is True when the graph has an arrow a → b.
`nx.descendants(G, v)` is every node you can reach from `v` by following arrows.
`nx.all_simple_paths(undirected, a, b)` lists every path from a to b that visits no node
twice; it runs on an undirected copy because a backdoor path may go against the arrows.
`itertools.combinations(pool, size)` gives every group of `size` names from `pool`.
`any(...)` is True if at least one item is, `all(...)` if every item is. The notes after a
colon or `->` (`G: nx.DiGraph`, `-> bool`) only say what type each argument and result is;
`set[str] | None` means a set of names, or None. `from __future__ import annotations` lets
those notes use this newer form on older versions of Python.

In [ ]:
%%writefile src/dag_utils.py
"""Graph utilities for choosing controls (ECON 5200 Lab 10).

"Which variables should I adjust for?" is a question about a graph, and once the
graph is stated the answer is computable. The graph itself is not computable:
it is an argument you have to make and defend.
"""
from __future__ import annotations

import itertools

import networkx as nx


def backdoor_paths(G: nx.DiGraph, treatment: str, outcome: str) -> list[list[str]]:
    """Every path from treatment to outcome that starts with an arrow INTO
    the treatment. These are the paths adjustment has to block."""
    undirected = G.to_undirected()
    out = []
    for path in nx.all_simple_paths(undirected, treatment, outcome):
        if len(path) > 1 and G.has_edge(path[1], treatment):
            out.append(path)
    return out


def is_collider(G: nx.DiGraph, path: list[str], i: int) -> bool:
    """Node i on the path has both neighbours pointing INTO it."""
    return G.has_edge(path[i - 1], path[i]) and G.has_edge(path[i + 1], path[i])


def path_blocked(G: nx.DiGraph, path: list[str], S: set[str]) -> bool:
    """Is this path blocked by conditioning on S?

    A non-collider in S blocks. A collider blocks BY DEFAULT and is opened
    by conditioning on it or on any of its descendants: the opposite of the
    rule for chains and forks.
    """
    for i in range(1, len(path) - 1):
        node = path[i]
        if is_collider(G, path, i):
            opened = node in S or any(d in S for d in nx.descendants(G, node))
            if not opened:
                return True          # collider left closed, so the path is blocked
        elif node in S:
            return True              # non-collider conditioned on, so blocked
    return False


def is_valid_adjustment_set(G: nx.DiGraph, treatment: str, outcome: str,
                            S: set[str]) -> bool:
    """Backdoor criterion: S blocks every backdoor path and contains no
    descendant of the treatment."""
    if any(v in nx.descendants(G, treatment) for v in S):
        return False
    return all(path_blocked(G, p, S)
               for p in backdoor_paths(G, treatment, outcome))


def minimal_adjustment_sets(G: nx.DiGraph, treatment: str, outcome: str,
                            observed: set[str] | None = None) -> list[set[str]]:
    """Every smallest valid adjustment set drawn from the observed variables.

    Returns [] when no set works: the effect is then not identified by
    adjustment on what you measured, and you need a different design
    (Chapters 11-13).
    """
    pool = sorted((observed if observed is not None
                   else set(G.nodes)) - {treatment, outcome})
    for size in range(len(pool) + 1):
        found = [set(c) for c in itertools.combinations(pool, size)
                 if is_valid_adjustment_set(G, treatment, outcome, set(c))]
        if found:
            return found
    return []

In [ ]:
# GIVEN — run as-is
# Import the module and test it on two small graphs
import importlib
import sys

sys.path.insert(0, "src")          # let Python find modules in the src folder
import dag_utils as du
importlib.reload(du)               # reload, so a re-run of the %%writefile cell takes effect

# The confounder case: U causes both D and Y. Each pair is one arrow.
G = nx.DiGraph([("U", "D"), ("U", "Y"), ("D", "Y")])
print("confounder, U observed :", du.minimal_adjustment_sets(G, "D", "Y", {"U"}))
print("confounder, U unobserved:", du.minimal_adjustment_sets(G, "D", "Y", set()))

# The collider case: D and Y both cause C
G2 = nx.DiGraph([("D", "C"), ("Y", "C")])
print("collider                :", du.minimal_adjustment_sets(G2, "D", "Y", {"C"}))

**Reading the output.** `[{'U'}]` says: adjust for U. `[]` says no set of the observed
variables works, so the effect is not identified by adjustment. `[set()]` says the empty set
works: adjust for nothing. The last two look alike and mean opposite things.

---
## AI-Assisted Expansion: an adjustment set proposed by AI, checked by you

Analysts now ask chatbots which controls to use, and the answer comes back fluent and
confident. Diagnoses 1-3 showed why that is not enough: whether a control helps depends on
the graph, not on how sensible the variable sounds. Here an AI proposes the controls for a
job-training study, and you check its set against the study's DAG with `dag_utils`, the
module you just ran. Proposing a set is quick work for an AI; checking it is yours.

**The setting.** The next cell simulates 20,000 people. Training raises skill by 1.5, and
earnings rise by 1.0 from training directly and by 2.0 per point of skill, as in Diagnosis 3,
so the true total effect of training on earnings is 4.0. How the data were made:
- `prior_earnings`: people who earned less before are more likely to enrol, and it raises later earnings.
- `drive` and `ability` are **not** in the data. `drive` makes a person more likely to enrol; `ability` raises later earnings.
- `risk_score`: recorded at enrolment, before anyone starts training, and computed from `drive` and `ability`.
- `skill`: a test everyone takes six months after enrolment; training raises it.
- `rating`: a supervisor's rating at the two-year follow-up; it depends on training and on earnings.

**What to do**
1. Copy **prompt version 1** below into Claude ([claude.northeastern.edu](https://claude.northeastern.edu)) or ChatGPT.
2. Read the answer before you check it. Type the AI's set into the version 1 cell as a
   list, and run the check below it. The prompt leaves one thing about the setting unsaid, so the AI had
   to assume something.
3. Find the problem against the DAG above, write **prompt version 2** stating what the AI
   was missing, and ask again in a new chat. Type the new set into the version 2 cell and run the check below it. Both checks stay
   in your notebook: questions 1 and 2 ask about each.
   If the AI's first set already passes, find where its reasoning assumed something the
   prompt did not say, and state that in version 2.
4. Answer the questions at the end. They are what is graded, not the AI's set.

**Reading the check.** It prints how many backdoor paths the set leaves open, whether
`dag_utils` calls the set valid, and the estimate with that set beside the true effect and
beside your own estimates from Diagnosis 3. It does not say which variable is wrong:
finding that is your job.

In [ ]:
# GIVEN — run as-is
# The setting: the study's DAG and its simulated data
# Its own generator, so this cell gives the same data however often you run it
job_rng = np.random.default_rng(RANDOM_STATE)
n_jobs = 20_000

drive = job_rng.normal(0, 1, n_jobs)          # not recorded
ability = job_rng.normal(0, 1, n_jobs)        # not recorded
prior_earnings = job_rng.normal(0, 1, n_jobs)
risk_score = 1.0 * drive + 1.0 * ability + job_rng.normal(0, 1, n_jobs)
# (... > 0) is True or False for each person; .astype(int) turns that into 1 or 0
training = (-1.0 * prior_earnings + 1.0 * drive + job_rng.normal(0, 1, n_jobs) > 0).astype(int)
skill = 1.5 * training + job_rng.normal(0, 1, n_jobs)
earnings = (1.0 * training + 2.0 * skill + 1.0 * prior_earnings + 1.0 * ability
            + job_rng.normal(0, 1, n_jobs))
rating = 1.2 * training + 1.2 * earnings + job_rng.normal(0, 1, n_jobs)
TRUE_EFFECT_JOBS = 1.0 + 2.0 * 1.5              # total effect, as in Diagnosis 3

# The data an analyst would have: drive and ability are missing.
# A dict of name: column pairs becomes a table with those column names.
jobs = pd.DataFrame({"training": training, "earnings": earnings,
                     "prior_earnings": prior_earnings, "risk_score": risk_score,
                     "skill": skill, "rating": rating})

# The DAG: each pair is one arrow, cause first
G_jobs = nx.DiGraph([
    ("prior_earnings", "training"), ("prior_earnings", "earnings"),
    ("drive", "training"), ("drive", "risk_score"),
    ("ability", "risk_score"), ("ability", "earnings"),
    ("training", "skill"), ("skill", "earnings"), ("training", "earnings"),
    ("training", "rating"), ("earnings", "rating"),
])

naive_jobs = sm.OLS(jobs["earnings"], sm.add_constant(jobs["training"])).fit().params["training"]
# {x:.1%} shows a share as a percent with one decimal
print(f"{len(jobs):,} people, {jobs['training'].mean():.1%} trained")
print(f"Columns: {list(jobs.columns)}")
print(f"earnings ~ training, no controls: {naive_jobs:.3f}   (true effect {TRUE_EFFECT_JOBS:.1f})")

In [ ]:
# GIVEN — run as-is
# The check, as a function: the cells below run it once for each version of the prompt
def check_ai_set(ai_set):
    """Judge one adjustment set against G_jobs and the jobs data."""
    # isinstance(x, list) is True when x is a list
    if not isinstance(ai_set, list):
        print("The set should be a list of column names in square brackets, each name in quotes.")
        return
    unknown = [v for v in ai_set if v not in jobs.columns]
    if unknown:
        print(f"Not columns in the data: {unknown}. The columns are {list(jobs.columns)}; "
              "use those names exactly.")
        return
    if "training" in ai_set or "earnings" in ai_set:
        print("The set holds controls only: leave out training and earnings.")
        return

    controls = sorted(set(ai_set))      # set() drops a name typed twice
    paths = du.backdoor_paths(G_jobs, "training", "earnings")
    # True counts as 1, so the sum is the number of paths left open
    still_open = sum(not du.path_blocked(G_jobs, p, set(controls)) for p in paths)
    valid = du.is_valid_adjustment_set(G_jobs, "training", "earnings", set(controls))
    fit = sm.OLS(jobs["earnings"], sm.add_constant(jobs[["training"] + controls])).fit()

    print(f"The AI's set: {controls}")
    print(f"Backdoor paths left open: {still_open} of {len(paths)}")
    print(f"Valid adjustment set (dag_utils): {valid}")
    print()
    print(f"Estimate with the AI's set:       {fit.params['training']:.3f}"
          f"   (SE {fit.bse['training']:.3f})")
    print(f"True total effect:                {TRUE_EFFECT_JOBS:.3f}")
    print(f"No controls (the setting cell):   {naive_jobs:.3f}")
    print(f"Diagnosis 3: Y ~ D {total_fit.params[1]:.3f}, Y ~ D + M {ctrl_fit.params[1]:.3f}")
    print("If the estimate is off, find which variable in the set is the problem "
          "before you trust the AI.")

In [ ]:
# PROMPT, version 1 — copy it exactly as it is
#
# I want the causal effect of a job-training programme on earnings, from
# observational data on 20,000 people, by regressing earnings on training
# plus a set of controls. The columns are:
#   training        1 if the person took the programme, 0 if not
#   earnings        earnings two years after enrolment
#   prior_earnings  earnings in the year before; people who earned less were
#                   more likely to enrol, and it also affects later earnings
#   risk_score      a caseworker's risk index, recorded at enrolment, before
#                   anyone started the programme
#   skill           a skills test everyone took six months after enrolment
#   rating          a supervisor's rating at the two-year follow-up
# Which columns should I control for? Give the answer as a Python list of
# column names, with one line of reasoning per column.

In [ ]:
# Type the AI's set from version 1 here as a list of column names, then run the check below.
# The form is: ai_set_v1 = ["column_a", "column_b"]

In [ ]:
# CHECK — the AI's version 1 set
if "ai_set_v1" not in globals():
    print("Type the AI's set in the cell above and run that cell first.")
else:
    check_ai_set(ai_set_v1)

In [ ]:
# PROMPT, version 2 — your revision. Keep everything from version 1 that was
# fine, and add or change what the AI was missing.
#
#

In [ ]:
# Type the AI's set from version 2 here as a list of column names, then run the check below.
# The form is: ai_set_v2 = ["column_a", "column_b"]

In [ ]:
# CHECK — the AI's version 2 set
if "ai_set_v2" not in globals():
    print("Type the AI's set in the cell above and run that cell first.")
else:
    check_ai_set(ai_set_v2)

### Document the interaction

This is what is graded for the expansion.

1. **Your revision.** What did you change between version 1 and version 2, and what did
   the AI get wrong, or have to assume, that made you change it? If its first set passed
   the check, name the assumption you made explicit.
2. **The check.** Which variable in the AI's first set was the problem, or which one was
   missing? Name the path in the DAG that shows it, and say whether that variable is a
   confounder, a collider, a mediator or a descendant of training. Did the estimate land
   near 4.0 after your revision? If the first set passed, name the path that including
   `risk_score` would have opened.
3. **In your own words.** Quote the AI's one line of reasoning for the variable it got
   wrong (or, if it got none wrong, for `risk_score`), and explain in 2–3 sentences what
   that reasoning leaves out.
4. **Would you use AI for this again?** One sentence: when is an AI's adjustment set worth
   asking for, and when must you draw the graph yourself first?

*Your answers here:*

1.
2.
3.
4.

---

## Extension (Optional — 45 min): How Wrong Could the Unobserved Confounder Make You?

Chapter 10's Tier B block gives the modern answer to "but what about unobserved confounding":
stop treating it as unanswerable and ask *how strong* a confounder would have to be to
overturn your result. Cinelli and Hazlett (2020) put that on a partial R-squared footing.

**Partial R-squared.** The partial R² of u with the outcome, given d, is the share of the
variation in y left over after regressing y on d that u explains.

In [ ]:
# YOUR TASK — optional: sensitivity analysis by simulation
# Build a DGP where one parameter sets an unobserved confounder's strength,
# estimate the effect WITHOUT it, and find the strength at which your
# conclusion changes.

TRUE_EFFECT = 0.5

def biased_estimate(conf_strength, n=20_000, seed=RANDOM_STATE):
    r = np.random.default_rng(seed)
    u = r.normal(0, 1, n)
    d = conf_strength * u + r.normal(0, 1, n)
    y = TRUE_EFFECT * d + conf_strength * u + r.normal(0, 1, n)
    # YOUR CODE: regress y on d ALONE (u is unobserved) and return the
    # coefficient on d, plus the partial R-squared of u with the outcome.
    ____

# YOUR CODE: sweep conf_strength from 0 to 2 and plot the estimate against
# the truth. Mark the strength at which the estimate doubles the true effect.

# YOUR CODE: write the sentence you would put in a paper: "an unobserved
# confounder would have to explain at least X% of the residual variation in
# the outcome to double this estimate."

### Interpretation Questions

1. Sensitivity analysis never proves the absence of confounding. Say precisely what it does establish, and why that is still worth reporting.
2. Your threshold is expressed in partial R-squared. Why is that a more useful currency for a referee than "I controlled for eight covariates"?
3. Compare this to the LaLonde exercise above. In one you have a benchmark; in the other you have a sensitivity bound. Which is the normal situation in applied work, and how should that shape what you claim?

*Your answers here:*

---

## Challenge (Take-Home): The Graph You Cannot Compute

The challenge is optional and ungraded.

`dag_utils.minimal_adjustment_sets` answers "given this graph, what should I adjust for?"
mechanically. It cannot tell you whether the graph is right.

In [ ]:
# YOUR TASK — optional, ungraded: take-home Challenge, argue a graph, then test it

# Step 1: pick an empirical question from your own field.

# Step 2: draw the DAG. Every arrow is a claim; write the claim in one line.

# Step 3: run minimal_adjustment_sets over the variables you could realistically
#   measure. Record what comes back, including if nothing does.

# Step 4: now draw a SECOND, equally defensible DAG that a competent critic
#   might propose, and run it again. Does the adjustment set change?

# Step 5: if the two graphs disagree, the data cannot arbitrate between them.
#   Say what evidence outside your dataset would, and what you would report
#   in the meantime.

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Diagnosed a regression in which adding a control caused by both the
  treatment and the outcome moved the estimate from [YOUR VALUE] to
  [YOUR VALUE] and raised R-squared to [YOUR VALUE], when the true effect
  was zero
* Showed that a control measured before treatment can still bias an
  estimate: [YOUR VALUE] against a true effect of zero
* Separated a total effect ([YOUR VALUE], truth 4.0) from a direct effect
  through a mediator ([YOUR VALUE], truth 1.0)
* Compared the LaLonde job-training experiment's benchmark of
  $[YOUR VALUE] with an observational estimate using CPS controls:
  $[YOUR VALUE] before covariate adjustment and $[YOUR VALUE] after
* Used a dag_utils.py module that finds valid adjustment sets from a DAG
* Had an AI propose the controls for a job-training study and checked its
  set against the study's DAG: its first set gave [YOUR VALUE] against a
  true effect of 4.0; after I revised my prompt once, [YOUR VALUE]
* (If I did the Extension) Found that an unobserved confounder explaining
  [YOUR VALUE]% of the residual variation in the outcome would double my
  estimate

**Please write a README.md entry including:**
1. Project Title: Causal Logic and DAGs: Diagnosing Bad Controls
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust',
'comprehensive' or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab10-causal-logic`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab10-causal-logic`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab10-causal-logic`, branch `main`,
   commit message `Lab 10 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. `dag_utils.py` is **not** pushed by step 2 — that step sends the notebook
   only, and the file lives in Colab's temporary storage, which is wiped when the
   session ends. Download it first: in the Colab **Files** panel (folder icon,
   left), open `src`, right-click `dag_utils.py` → **Download**. Then on the
   repo page use **Add file → Upload files**, drag it in, and **Commit changes**.
   It is part of what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.